In [15]:
import argparse
import sys
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from skimage import io
import os
import torch
from pytorch_fid import fid_score

In [16]:
ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
INPAINTED_DIR = os.path.join(ROOT_DIR, 'results', 'inpainted')
DATA_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched')

dirs = ['Test']

patch_types = [
    'Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6',
    'TSEA1', 'TSEA2'
]

In [18]:
for d in dirs:
    for patch in patch_types:
        path = os.path.join(DATA_DIR, d, patch, 'images')
        ssim_scores = []
        for root, _, files in os.walk(path):
            for file in files:
                if not file.lower().endswith('.jpg'):
                    continue
                impath = os.path.join(path, file)
                img_inpainted = io.imread(os.path.join(INPAINTED_DIR, file.replace(".jpg", "_mask_xgb.png_comp.png")))
                img_patched = io.imread(impath)
                    
                h, w = img_inpainted.shape[:2]
                img_patched = cv2.resize(img_patched, (w, h), interpolation=cv2.INTER_AREA)
                img_inpainted = img_inpainted.astype(np.float32)
                img_patched = img_patched.astype(np.float32)
                
                ssim_scores.append(ssim(img_inpainted, img_patched, data_range=img_inpainted.max() - img_inpainted.min(), channel_axis=-1))
        ssim_avg = sum(ssim_scores) / len(ssim_scores)
        print(f"Avg SSIM (Inpainted) Score for {patch} patch: {ssim_avg}")

Avg SSIM Score for Naturalistic1 patch: 0.9707797169685364
Avg SSIM Score for Naturalistic2 patch: 0.9674369692802429
Avg SSIM Score for Naturalistic3 patch: 0.9717115163803101
Avg SSIM Score for Naturalistic4 patch: 0.9702999591827393
Avg SSIM Score for Naturalistic5 patch: 0.9711973071098328
Avg SSIM Score for Naturalistic6 patch: 0.9731384515762329
Avg SSIM Score for TSEA1 patch: 0.9713727831840515
Avg SSIM Score for TSEA2 patch: 0.9689871668815613


In [19]:
dir0 = os.path.join(ROOT_DIR, 'results', 'inpainted_patch')
for patch in patch_types:
    path = os.path.join(dir0, patch)
    for root, _, files in os.walk(path):
        for file in files:
            if not file.lower().endswith('.png'):
                continue
            impath = os.path.join(path, file)
            new_file = impath.replace("_mask_xgb.png_comp.png", ".png")
            os.rename(impath, new_file)

In [22]:
for d in dirs:
    for patch in patch_types:
        path = os.path.join(DATA_DIR, 'Clean_Test', patch, 'images')
        out_dir = os.path.join(DATA_DIR, 'resized_clean', patch)
        os.makedirs(out_dir, exist_ok=True)

        ssim_scores = []

        for root, _, files in os.walk(path):
            for file in files:
                if not file.lower().endswith('.jpg'):
                    continue

                impath = os.path.join(root, file)
                img = cv2.imread(impath)

                if img is None:
                    print(f"Could not read {impath}")
                    continue

                # Resize to 512x512
                resized = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)

                # Save as PNG
                out_name = patch + "_" + os.path.splitext(file)[0] + ".png"
                out_path = os.path.join(out_dir, out_name)
                cv2.imwrite(out_path, resized)

                print(f"Saved resized image: {out_path}")

Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\resized_clean\Naturalistic1\Naturalistic1_1499917703635.png
Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\resized_clean\Naturalistic1\Naturalistic1_1499918339461.png
Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\resized_clean\Naturalistic1\Naturalistic1_1499923500376.png
Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\resized_clean\Naturalistic1\Naturalistic1_1499925371739.png
Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\resized_clean\Naturalistic1\Naturalistic1_1499931818829.png
Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\resized_clean\Naturalistic1\Naturalistic1_1499932367119.png
Saved resized image: C:\Adrianov\Projects\Project-Satanael\data\tju-dhd\eval_final_patched\res